In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 The KV Cache: How a Brilliant Trick Moves the Bottleneck

*Part of the Vizuara series on LLM Systems*
*Estimated time: 25-30 minutes*

---

Welcome! In this notebook we are going to pull back the curtain on one of the most important engineering tricks in modern language model inference — the **Key-Value Cache**. By the end, you will understand not just *what* it does, but *why* it was necessary, *how* it works at the level of arithmetic, and *what price* it extracts from your hardware.

Let us get the tools we need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import time
import torch

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Pretty plots
plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor": "#1a1a2e",
    "axes.edgecolor": "#444",
    "axes.labelcolor": "white",
    "xtick.color": "white",
    "ytick.color": "white",
    "text.color": "white",
    "grid.color": "#333",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
    "legend.facecolor": "#1a1a2e",
    "legend.edgecolor": "#555",
    "figure.titlesize": 14,
})

ACCENT  = "#7b2ff7"   # purple
ACCENT2 = "#00d4ff"   # cyan
WARN    = "#ff6b6b"   # red
OK      = "#51cf66"   # green
YELLOW  = "#ffd43b"   # yellow

print("✅ Environment ready. Let us begin!")
print(f"   PyTorch  : {torch.__version__}")
print(f"   Device   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Why Does This Matter?

Imagine you are using ChatGPT to help you draft a legal contract. You paste in 2,000 words of context and ask the model to continue. The model generates one word, then another, then another — streaming text onto your screen in real time.

**Every single one of those words is a separate forward pass through a billion-parameter neural network.**

Now here is the uncomfortable question: when generating word number 2,001, does the model need to re-read all 2,000 previous words? In a naive implementation — yes, completely. For word 2,002 — yes again, all 2,001 words. The cost compounds with every step.

This is not a software bug. It is the natural consequence of how the **attention mechanism** works. Let us feel the pain of this before we appreciate the cure.

In [ ]:
# ── Let us visualise the COST of naive inference vs. the KV-cached version ──

seq_lengths   = np.arange(1, 2001, 50)

# Naive: to generate token t, we recompute attention over ALL t tokens.
# Total FLOPs ∝ sum_{t=1}^{T} t^2  (quadratic per step, summed over steps)
naive_flops = np.array([np.sum(np.arange(1, T+1)**2) for T in seq_lengths])

# KV-Cache: we only compute attention of the NEW token against cached keys.
# Total FLOPs ∝ sum_{t=1}^{T} t  (linear per step, summed over steps)
cached_flops = np.array([np.sum(np.arange(1, T+1)) for T in seq_lengths])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Naive vs. KV-Cached Inference — The Scale of the Difference",
             fontsize=13, color="white", y=1.02)

for ax, ys, titles, colors in zip(
        axes,
        [(naive_flops, cached_flops), (naive_flops / cached_flops,)],
        [("Naive (re-compute everything)", "KV-Cached"), ("Speed-up ratio (naive / cached)",)],
        [(WARN, OK), (YELLOW,)]):
    for y, title, color in zip(ys, titles, colors):
        ax.plot(seq_lengths, y / 1e9, label=title, color=color, linewidth=2)
    ax.set_xlabel("Context length (tokens)")
    ax.set_ylabel("Relative compute units (×10⁹)")
    ax.legend()
    ax.grid(True)

fig.tight_layout()
plt.savefig("cost_comparison.png", dpi=140, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("📊 The right panel shows the KV cache delivers a speed-up that grows with context length.")
print("   At 2,000 tokens the cached version does ~1,000× less arithmetic. Remarkable.")

## 2. Building Intuition

Before we write a single equation, let us build a crystal-clear mental model.

### 2.1 The Attention Mechanism in Plain English

Attention answers one question: *"Given the token I am currently producing, how much should I look at each past token?"*

To answer this, every token in the sequence gets three vectors assigned to it:
- A **Query (Q)** — what this token is *asking for*
- A **Key (K)** — what this token *advertises* about itself
- A **Value (V)** — what this token *contributes* if selected

When generating a new token, we compute the dot product between the new token's **Query** and every past token's **Key**. High dot product → high attention weight → that past token's **Value** gets mixed in heavily.

### 2.2 The Repetition Problem

Here is the key observation: **Keys and Values for past tokens never change.**

In naive inference, every time we generate a new token, we:
1. Re-project *all* past tokens through weight matrices to get their K and V vectors
2. Compute dot products between the new query and all those K vectors
3. Weighted-sum the V vectors

Steps 1 is pure waste. Those K and V vectors were identical the step before. And the step before that.

### 2.3 The KV Cache Fix

The fix is beautifully simple: **store the K and V vectors the first time you compute them, and look them up instead of recomputing.**

Think of it like a professor writing lecture notes. Without a cache, the professor re-derives every theorem from scratch before each class. With a cache, the professor writes notes once and just reads from them. The *reading* still takes time — but the expensive *derivation* happens only once.

### 2.4 The Hidden Cost — Memory Bandwidth

Here is where the story gets interesting. Once we skip all that repeated arithmetic, what is left? We still need to *read* the cached K and V tensors from memory on every generation step.

For long sequences, this memory-reading becomes the new bottleneck. The GPU chip sits mostly idle — not because it lacks arithmetic to do, but because it is **waiting for data to arrive from VRAM**. This is the regime engineers call **memory-bound**.

### 🤔 Think About This

> A GPU can do ~300 **trillion** arithmetic operations per second, but can only move ~2 **terabytes** of data per second from memory. If your computation requires reading 10 GB of cached K/V tensors to do only 1 million multiplications, which resource runs out first — arithmetic capacity or memory bandwidth? What does that mean for how fast your model actually runs?

Keep that question in mind as we work through the math.

## 3. The Mathematics

### 3.1 Scaled Dot-Product Attention

For a sequence of $T$ tokens, each with embedding dimension $d$, the attention output is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

where $Q, K, V \in \mathbb{R}^{T \times d_k}$.

**What this equation says:** Take each query row, dot it with every key row (giving a $T \times T$ score matrix), scale by $\sqrt{d_k}$ to prevent extreme softmax saturation, normalise into probabilities, then use those probabilities to blend the value rows.

**Computationally, this means:** The $QK^\top$ multiplication alone costs $\mathcal{O}(T^2 \cdot d_k)$ operations — quadratic in sequence length.

### 3.2 The Cost of Naive Auto-Regressive Decoding

When generating token $t$, the full context has length $t$. The cost of one attention forward pass is:

$$\text{FLOPs}_{\text{naive}}(t) \approx 4 \cdot t^2 \cdot d_k$$

*(the factor of 4 accounts for Q, K, V projections plus the attention matmul)*

Over a generation of $T$ tokens from a prompt of length $P$:

$$\text{Total FLOPs}_{\text{naive}} = \sum_{t=P}^{P+T} 4 \cdot t^2 \cdot d_k \approx \frac{4 d_k}{3}\left[(P+T)^3 - P^3\right]$$

**This is cubic in sequence length.** At $T=2000$ tokens, this grows astronomically fast.

### 3.3 The Cost With a KV Cache

With a KV cache, at generation step $t$ we only need to compute the new token's $Q$, $K$, $V$ vectors (cost $\propto d_k$) and dot the new $Q$ against all $t$ cached $K$s:

$$\text{FLOPs}_{\text{cached}}(t) \approx 4 \cdot t \cdot d_k$$

Over the same generation:

$$\text{Total FLOPs}_{\text{cached}} = \sum_{t=P}^{P+T} 4 \cdot t \cdot d_k \approx 2 d_k \left[(P+T)^2 - P^2\right]$$

**Quadratic instead of cubic** — a savings ratio that grows linearly with context length.

### 3.4 The Roofline Model

The roofline model characterises hardware performance with one key quantity — **arithmetic intensity** $I$:

$$I = \frac{\text{FLOPs}}{\text{Bytes moved from memory}}$$

The achieved performance $P$ is then bounded by:

$$P \leq \min\!\left(\pi,\ \beta \cdot I\right)$$

where $\pi$ is the chip's **peak compute throughput** (FLOPs/s) and $\beta$ is the **memory bandwidth** (bytes/s).

**This equation says:** performance is the minimum of two ceilings. Below the *ridge point* $I^* = \pi / \beta$, we are memory-bound and performance scales with $I$. Above it, we are compute-bound and performance is flat at $\pi$.

**The KV cache's effect:** By slashing FLOPs without reducing the bytes we must read (we still fetch all cached K, V tensors), it *decreases* arithmetic intensity — pushing us deeper into the memory-bound regime.

In [ ]:
# ── Visualise the Roofline Model ──

fig, ax = plt.subplots(figsize=(10, 5.5))

# Hardware numbers (approximate A100 SXM4 in FP16)
beta_GBps = 2_000          # GB/s memory bandwidth
pi_TFLOPs = 312            # TFLOP/s compute (FP16)
ridge_point = (pi_TFLOPs * 1e12) / (beta_GBps * 1e9)   # ops/byte ≈ 156

I_vals = np.logspace(-1, 4, 400)    # arithmetic intensity axis
roofline = np.minimum(pi_TFLOPs * np.ones_like(I_vals),
                      beta_GBps * 1e-3 * I_vals)   # TFLOP/s units

ax.plot(I_vals, roofline, color=ACCENT2, linewidth=3, label="Roofline (A100 FP16)")
ax.axvline(ridge_point, color=YELLOW, linewidth=1.5, linestyle="--",
           label=f"Ridge point  (I* ≈ {ridge_point:.0f} ops/byte)")

# Shade regions
ax.fill_between(I_vals, 0, roofline,
                where=(I_vals < ridge_point), alpha=0.15, color=WARN,
                label="Memory-Bound region")
ax.fill_between(I_vals, 0, roofline,
                where=(I_vals >= ridge_point), alpha=0.15, color=OK,
                label="Compute-Bound region")

# Mark LLM inference during generation (typical I ~ 1-10 ops/byte)
ax.scatter([3], [beta_GBps * 3 * 1e-3], color=WARN, s=200, zorder=5,
           label="LLM generation (KV-cached)", marker="★")
ax.annotate("We live here\n(memory-bound!)", xy=(3, beta_GBps * 3 * 1e-3),
            xytext=(12, 4), color=WARN, fontsize=9,
            arrowprops=dict(arrowstyle="->", color=WARN))

ax.set_xscale("log")
ax.set_xlabel("Arithmetic Intensity  I  [ops / byte]", fontsize=11)
ax.set_ylabel("Achieved Performance  [TFLOP/s]", fontsize=11)
ax.set_title("The Roofline Model — Where Does LLM Inference Live?", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True)
fig.tight_layout()
plt.savefig("roofline.png", dpi=140, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("📊 Notice that LLM generation sits far left of the ridge point.")
print("   The GPU is not compute-starved — it is data-starved.")

## 4. Let's Build It — Component by Component

Let us implement everything from scratch, using only basic PyTorch operations. No `nn.MultiheadAttention` black boxes. We want to see every tensor move.

### 4.1 A Single Attention Head (No Cache)

We start with the most basic building block: vanilla scaled dot-product attention for a complete sequence.

In [ ]:
def attention_no_cache(Q: torch.Tensor,
                       K: torch.Tensor,
                       V: torch.Tensor,
                       mask: bool = True) -> torch.Tensor:
    """
    Vanilla causal scaled dot-product attention for a full sequence.

    Args:
        Q : (T, d_k)  — query matrix
        K : (T, d_k)  — key   matrix
        V : (T, d_v)  — value matrix
        mask: apply causal mask so token t cannot see tokens > t

    Returns:
        out : (T, d_v) — attended output for every position
    """
    T, d_k = Q.shape

    # Step 1: raw attention scores  (T, T)
    scores = Q @ K.T / (d_k ** 0.5)

    # Step 2: causal mask — upper triangle gets -inf so softmax ≈ 0
    if mask:
        causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool,
                                            device=Q.device), diagonal=1)
        scores = scores.masked_fill(causal_mask, float("-inf"))

    # Step 3: softmax → weights
    weights = torch.softmax(scores, dim=-1)   # (T, T)

    # Step 4: weighted sum of values
    out = weights @ V                          # (T, d_v)
    return out

# Quick sanity check
T, d_k, d_v = 8, 16, 16
Q_test = torch.randn(T, d_k)
K_test = torch.randn(T, d_k)
V_test = torch.randn(T, d_v)

out_test = attention_no_cache(Q_test, K_test, V_test)
print(f"✅ attention_no_cache output shape: {out_test.shape}  (expected: ({T}, {d_v}))")

Now let us visualise the attention weight matrix so we can see the causal mask working:

In [ ]:
# ── Visualise the attention weight matrix ──
T_vis, d_k_vis = 12, 32
Q_vis = torch.randn(T_vis, d_k_vis)
K_vis = torch.randn(T_vis, d_k_vis)
V_vis = torch.randn(T_vis, d_k_vis)

scores_vis = Q_vis @ K_vis.T / (d_k_vis ** 0.5)
causal_mask = torch.triu(torch.ones(T_vis, T_vis, dtype=torch.bool), diagonal=1)
scores_vis = scores_vis.masked_fill(causal_mask, float("-inf"))
weights_vis = torch.softmax(scores_vis, dim=-1).detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

im0 = axes[0].imshow(weights_vis, cmap="magma", aspect="auto")
axes[0].set_title("Attention Weights (causal mask applied)", fontsize=11)
axes[0].set_xlabel("Key position (past tokens)")
axes[0].set_ylabel("Query position (current token)")
plt.colorbar(im0, ax=axes[0])

# Show which cells are masked vs. active
mask_display = causal_mask.float().numpy()
im1 = axes[1].imshow(mask_display, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=1)
axes[1].set_title("Causal Mask  (red = blocked, green = allowed)", fontsize=11)
axes[1].set_xlabel("Key position")
axes[1].set_ylabel("Query position")
plt.colorbar(im1, ax=axes[1])

fig.tight_layout()
plt.savefig("attention_weights.png", dpi=140, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("📊 Every row represents one query token. It can only attend to tokens at or before its position.")

### 4.2 The KV Cache Data Structure

Now let us build the cache. The idea is simple: a dictionary (or class) that accumulates K and V vectors as we generate, one step at a time.

In [ ]:
class KVCache:
    """
    A minimal Key-Value cache for a single attention head.

    The cache stores tensors of shape (t, d_k) and (t, d_v) where t
    grows by 1 on every generation step.
    """

    def __init__(self):
        self.K_cache = None   # (t, d_k)
        self.V_cache = None   # (t, d_v)

    def update(self, k_new: torch.Tensor, v_new: torch.Tensor):
        """
        Append one new (key, value) pair to the cache.

        Args:
            k_new : (1, d_k)  — key for the newly generated token
            v_new : (1, d_v)  — value for the newly generated token
        """
        if self.K_cache is None:
            self.K_cache = k_new
            self.V_cache = v_new
        else:
            self.K_cache = torch.cat([self.K_cache, k_new], dim=0)
            self.V_cache = torch.cat([self.V_cache, v_new], dim=0)

    def size(self):
        return 0 if self.K_cache is None else self.K_cache.shape[0]

    def memory_bytes(self, dtype_bytes: int = 2):
        """Estimate VRAM consumed by this cache (bytes)."""
        if self.K_cache is None:
            return 0
        return (self.K_cache.numel() + self.V_cache.numel()) * dtype_bytes

    def reset(self):
        self.K_cache = None
        self.V_cache = None

# Test the cache
cache = KVCache()
d_k, d_v = 64, 64
for step in range(5):
    k = torch.randn(1, d_k)
    v = torch.randn(1, d_v)
    cache.update(k, v)

print(f"✅ Cache size after 5 steps: {cache.size()} tokens")
print(f"   K_cache shape : {cache.K_cache.shape}")
print(f"   V_cache shape : {cache.V_cache.shape}")
print(f"   Memory used   : {cache.memory_bytes():,} bytes  ({cache.memory_bytes()/1024:.1f} KB)")

### 4.3 Cached Attention — One New Token at a Time

Now we wire the cache into the attention function. At each generation step, we only have **one** new query. We look it up against **all** cached keys.

In [ ]:
def attention_with_cache(q_new: torch.Tensor,
                         k_new: torch.Tensor,
                         v_new: torch.Tensor,
                         cache: KVCache) -> tuple[torch.Tensor, KVCache]:
    """
    Compute attention for a single new token using the KV cache.

    Args:
        q_new  : (1, d_k)  — query for the new token
        k_new  : (1, d_k)  — key   for the new token
        v_new  : (1, d_v)  — value for the new token
        cache  : KVCache   — previously stored K and V tensors

    Returns:
        out    : (1, d_v)  — output for the new token
        cache  : KVCache   — updated cache (new token appended)
    """
    # Step 1: add new token to cache
    cache.update(k_new, v_new)

    # Step 2: the new query attends to ALL cached keys  (1, t)
    d_k = q_new.shape[-1]
    scores = q_new @ cache.K_cache.T / (d_k ** 0.5)   # (1, t)

    # No mask needed — by construction all cached tokens are in the past
    weights = torch.softmax(scores, dim=-1)            # (1, t)

    # Step 3: blend cached values
    out = weights @ cache.V_cache                      # (1, d_v)
    return out, cache


# ── End-to-end correctness check ──
# Generate a sequence token by token using the cache,
# then compare to the naive full-sequence attention at the final step.

T_check, d_k, d_v = 10, 32, 32
# Pretend we have fixed projection matrices (identity for simplicity)
tokens = torch.randn(T_check, d_k)   # raw token embeddings = Q = K = V

# --- Cached path ---
cache_check = KVCache()
outputs_cached = []
for t in range(T_check):
    q_t = tokens[t:t+1]
    out_t, cache_check = attention_with_cache(q_t, q_t.clone(), q_t.clone(), cache_check)
    outputs_cached.append(out_t)

final_cached = outputs_cached[-1]   # output for last token

# --- Naive path (full sequence) ---
final_naive = attention_no_cache(tokens, tokens, tokens)[-1:]

# They should match (up to floating point noise)
max_diff = (final_cached - final_naive).abs().max().item()
print(f"✅ Max difference between cached and naive output: {max_diff:.2e}")
print(f"   {'PASS ✓' if max_diff < 1e-5 else 'FAIL ✗ — something is wrong!'}")

In [ ]:
# 📊 Visualise how the cache grows over generation steps

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

steps = np.arange(1, T_check + 1)
cache_mem_kb = steps * 2 * d_k * 2 / 1024   # 2 matrices (K,V), d_k floats, 2 bytes each

axes[0].bar(steps, cache_mem_kb, color=ACCENT, alpha=0.85, edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Generation step (token index)")
axes[0].set_ylabel("KV Cache size (KB)")
axes[0].set_title("Cache Memory Grows Linearly with Sequence Length")
axes[0].grid(True, axis="y")

# Show attention score vectors growing wider each step
all_weights = []
cache_vis = KVCache()
for t in range(T_check):
    q_t = tokens[t:t+1]
    cache_vis.update(q_t.clone(), q_t.clone())
    scores_t = q_t @ cache_vis.K_cache.T / (d_k ** 0.5)
    w_t = torch.softmax(scores_t, dim=-1).squeeze().detach().numpy()
    padded = np.pad(w_t, (0, T_check - len(w_t)))
    all_weights.append(padded)

im = axes[1].imshow(np.array(all_weights), cmap="viridis", aspect="auto")
axes[1].set_xlabel("Token position in cache")
axes[1].set_ylabel("Generation step")
axes[1].set_title("Attention Weights Over Cache  (each row = one step)")
plt.colorbar(im, ax=axes[1])

fig.tight_layout()
plt.savefig("cache_growth.png", dpi=140, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

## 5. 🔧 Your Turn

Now it is your turn to cement the understanding. We are going to implement two things:

1. **`compute_arithmetic_intensity`** — compute the roofline model's key quantity $I$ for a given generation step
2. **`compare_flops`** — measure and compare the theoretical FLOPs for both inference modes

Take your time. The hints walk you through it step by step.

In [ ]:
def compute_arithmetic_intensity(seq_len: int,
                                 d_model: int,
                                 d_k: int,
                                 use_cache: bool,
                                 dtype_bytes: int = 2) -> dict:
    """
    Estimate arithmetic intensity for one attention forward pass.

    Arithmetic Intensity I = FLOPs / Bytes_moved_from_memory

    Args:
        seq_len    : current context length (number of tokens)
        d_model    : model embedding dimension
        d_k        : key/query dimension per head
        use_cache  : if True, assume KV cache is active (only new token computed)
        dtype_bytes: bytes per parameter (2 for FP16, 4 for FP32)

    Returns:
        A dict with keys: 'flops', 'bytes', 'intensity'
    """
    # ============ TODO ============
    # Step 1: Estimate FLOPs for the attention matmul.
    #   - Without cache: we compute Q @ K^T for a (seq_len × seq_len) matrix
    #     Each element is a dot product of length d_k → costs 2*d_k FLOPs
    #     Total: 2 * seq_len * seq_len * d_k
    #   - With cache: only ONE new query attends to all seq_len keys
    #     Total: 2 * 1 * seq_len * d_k
    #
    # Step 2: Estimate bytes moved from memory.
    #   - Without cache: we must load Q, K, V matrices of shape (seq_len, d_k)
    #     Bytes = 3 * seq_len * d_k * dtype_bytes
    #   - With cache: we load the full K and V caches (seq_len, d_k each)
    #     plus one new Q row. Bytes ≈ 2 * seq_len * d_k * dtype_bytes
    #
    # Step 3: Compute intensity = flops / bytes
    #
    # ============ END TODO ========
    flops  = ???   # YOUR CODE HERE
    bytes_ = ???   # YOUR CODE HERE
    return {
        "flops"    : flops,
        "bytes"    : bytes_,
        "intensity": flops / bytes_,
    }


# ── Verification cell ──
result_naive  = compute_arithmetic_intensity(512, 768, 64, use_cache=False)
result_cached = compute_arithmetic_intensity(512, 768, 64, use_cache=True)

assert result_naive["flops"]  > result_cached["flops"],  "Naive should have more FLOPs"
assert result_naive["intensity"] > result_cached["intensity"], \
    "Cached mode should have LOWER arithmetic intensity (more memory-bound)"

print("✅ Verification passed!")
print(f"\n  Sequence length : 512 tokens,  d_k = 64")
print(f"  {'Mode':<12}  {'FLOPs':>15}  {'Bytes':>15}  {'Intensity':>12}")
print(f"  {'-'*60}")
for label, res in [("Naive", result_naive), ("KV-Cached", result_cached)]:
    print(f"  {label:<12}  {res['flops']:>15,.0f}  {res['bytes']:>15,.0f}  "
          f"{res['intensity']:>11.2f}")
print(f"\n  The cached version has {result_naive['intensity']/result_cached['intensity']:.1f}× "
      f"lower arithmetic intensity → deeper in the memory-bound zone.")

Now implement the wall-clock timing comparison:

In [ ]:
def time_inference_step(seq_len: int,
                        d_k: int,
                        device: str = "cpu",
                        n_trials: int = 50) -> dict:
    """
    Time one generation step for both naive and cached attention.

    Args:
        seq_len  : current context length
        d_k      : key dimension
        device   : 'cpu' or 'cuda'
        n_trials : number of timing repetitions

    Returns:
        dict with keys 'naive_ms' and 'cached_ms' (milliseconds, mean)
    """
    # ============ TODO ============
    # Step 1: Create random Q, K, V tensors of shape (seq_len, d_k) on device.
    #
    # Step 2: Time the NAIVE path:
    #   Call attention_no_cache(Q, K, V) n_trials times.
    #   Record the elapsed wall-clock time per call in milliseconds.
    #   Tip: use time.perf_counter() before and after.
    #   If device == 'cuda', call torch.cuda.synchronize() before timing reads.
    #
    # Step 3: Time the CACHED path:
    #   For each trial, build a fresh KVCache pre-filled with (seq_len-1) entries,
    #   then call attention_with_cache with one new token.
    #   This simulates generating step seq_len given a full context.
    #
    # Step 4: Return mean times.
    #
    # ============ END TODO ========
    naive_times  = []
    cached_times = []
    # YOUR CODE HERE
    return {
        "naive_ms" : np.mean(naive_times),
        "cached_ms": np.mean(cached_times),
    }


# Quick check on small input
result = time_inference_step(seq_len=64, d_k=32, n_trials=10)
assert "naive_ms"  in result and result["naive_ms"]  > 0
assert "cached_ms" in result and result["cached_ms"] > 0
print("✅ time_inference_step structure looks good!")
print(f"   naive: {result['naive_ms']:.3f} ms   cached: {result['cached_ms']:.3f} ms")

## 6. Putting It All Together

Let us now sweep over sequence lengths and measure everything end-to-end. We will combine the theoretical arithmetic intensity analysis with the empirical timing data.

In [ ]:
# ── Sweep over sequence lengths ──

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device.upper()}")

seq_lengths_sweep = [32, 64, 128, 256, 512, 768, 1024]
d_k_sweep = 64
d_model_sweep = 512
n_trials = 30

results = []
for T in seq_lengths_sweep:
    print(f"  seq_len={T:5d} ...", end="", flush=True)

    # Theoretical intensity
    ai_naive  = compute_arithmetic_intensity(T, d_model_sweep, d_k_sweep,
                                             use_cache=False)
    ai_cached = compute_arithmetic_intensity(T, d_model_sweep, d_k_sweep,
                                             use_cache=True)

    # Empirical timing
    timing = time_inference_step(T, d_k_sweep, device=device, n_trials=n_trials)

    results.append({
        "T"              : T,
        "naive_flops"    : ai_naive["flops"],
        "cached_flops"   : ai_cached["flops"],
        "naive_intensity": ai_naive["intensity"],
        "cached_intensity": ai_cached["intensity"],
        "naive_ms"       : timing["naive_ms"],
        "cached_ms"      : timing["cached_ms"],
        "speedup"        : timing["naive_ms"] / timing["cached_ms"],
    })
    print(f" naive={timing['naive_ms']:.2f}ms  cached={timing['cached_ms']:.2f}ms  "
          f"speedup={timing['naive_ms']/timing['cached_ms']:.1f}×")

print("\n✅ Sweep complete!")

In [ ]:
# ── Comprehensive results visualisation ──

T_vals      = [r["T"]               for r in results]
speedups    = [r["speedup"]         for r in results]
naive_ms    = [r["naive_ms"]        for r in results]
cached_ms   = [r["cached_ms"]       for r in results]
naive_I     = [r["naive_intensity"] for r in results]
cached_I    = [r["cached_intensity"]for r in results]

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

# ── Panel 1: Wall-clock time ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(T_vals, naive_ms,  "o-", color=WARN,  linewidth=2, label="Naive (re-compute)")
ax1.plot(T_vals, cached_ms, "s-", color=OK,    linewidth=2, label="KV-Cached")
ax1.set_xlabel("Context length (tokens)")
ax1.set_ylabel("Wall-clock time per step (ms)")
ax1.set_title("⏱  Latency Comparison")
ax1.legend(fontsize=8)
ax1.grid(True)

# ── Panel 2: Speed-up ratio ──
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(T_vals, speedups, color=ACCENT, alpha=0.85, edgecolor="white")
ax2.axhline(1.0, color="white", linewidth=1, linestyle="--")
ax2.set_xlabel("Context length (tokens)")
ax2.set_ylabel("Speed-up  (naive / cached)")
ax2.set_title("🚀  KV Cache Speed-up")
ax2.grid(True, axis="y")
for x, y in zip(T_vals, speedups):
    ax2.text(x, y + 0.05, f"{y:.1f}×", ha="center", va="bottom",
             fontsize=7.5, color="white")

# ── Panel 3: Arithmetic intensity ──
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(T_vals, naive_I,  "o-", color=WARN,  linewidth=2, label="Naive")
ax3.plot(T_vals, cached_I, "s-", color=ACCENT2, linewidth=2, label="KV-Cached")
ax3.axhline(156, color=YELLOW, linewidth=1.5, linestyle="--",
            label="Ridge point ~156 ops/byte")
ax3.set_xlabel("Context length (tokens)")
ax3.set_ylabel("Arithmetic Intensity (ops/byte)")
ax3.set_title("📉  Intensity Drops With Cache")
ax3.legend(fontsize=8)
ax3.grid(True)

# ── Panel 4: VRAM growth ──
ax4 = fig.add_subplot(gs[1, 1])
# KV cache VRAM = 2 (K and V) × T × d_k × 2 bytes (FP16)
kv_vram_mb = [2 * T * d_k_sweep * 2 / (1024**2) for T in T_vals]
# Scale to realistic model (e.g. 32 heads, 32 layers)
n_heads, n_layers = 32, 32
kv_vram_full_gb = [v * n_heads * n_layers / 1024 for v in kv_vram_mb]
ax4.plot(T_vals, kv_vram_full_gb, "^-", color=YELLOW, linewidth=2)
ax4.set_xlabel("Context length (tokens)")
ax4.set_ylabel("KV Cache VRAM (GB)\n[32 heads × 32 layers]")
ax4.set_title("💾  Memory Cost of the Cache")
ax4.grid(True)
ax4.fill_between(T_vals, 0, kv_vram_full_gb, alpha=0.2, color=YELLOW)

fig.suptitle("KV Cache: Speed Wins vs. Memory Costs — The Full Picture",
             fontsize=13, y=1.01, color="white")
plt.savefig("full_analysis.png", dpi=140, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

## 7. The Memory Bandwidth Bottleneck — Made Concrete

Let us do one final, grounded calculation. How long does it *actually* take to just **read** the KV cache off a real GPU, ignoring all arithmetic?

In [ ]:
# ── Bandwidth-bound time estimation ──
# This shows that for long contexts, the KV cache read itself
# becomes the dominant cost — before we even do any arithmetic.

print("=" * 62)
print("  Bandwidth Analysis: How Long Does Reading the KV Cache Take?")
print("=" * 62)

# A100 SXM4 specs
BANDWIDTH_GBps  = 2_000   # GB/s
PEAK_TFLOPS     = 312     # TFLOP/s (FP16)
RIDGE_OPS_BYTE  = (PEAK_TFLOPS * 1e12) / (BANDWIDTH_GBps * 1e9)

# GPT-3 scale model: 96 layers, 96 heads, d_k = 128
n_layers_gpt3 = 96
n_heads_gpt3  = 96
d_k_gpt3      = 128
dtype_bytes   = 2          # FP16

print(f"\n  Model   : GPT-3 scale  (96 layers, 96 heads, d_k = {d_k_gpt3})")
print(f"  GPU     : A100 SXM4  bandwidth = {BANDWIDTH_GBps} GB/s")
print(f"  Ridge pt: {RIDGE_OPS_BYTE:.0f} ops/byte\n")
print(f"  {'Ctx len':>8}  {'KV Cache':>10}  {'Read time':>12}  {'Intensity':>12}  {'Regime'}")
print(f"  {'-'*60}")

context_lengths = [128, 512, 1024, 2048, 4096, 8192]
for T in context_lengths:
    # Total bytes in the KV cache for the full model
    kv_bytes = 2 * T * d_k_gpt3 * n_heads_gpt3 * n_layers_gpt3 * dtype_bytes
    kv_gb    = kv_bytes / 1e9

    # Time to just stream this from memory
    read_time_ms = (kv_bytes / (BANDWIDTH_GBps * 1e9)) * 1000

    # Arithmetic intensity for one generation step at this context length
    flops_step = 2 * T * d_k_gpt3 * n_heads_gpt3 * n_layers_gpt3
    intensity  = flops_step / kv_bytes

    regime = "✅ compute-bound" if intensity > RIDGE_OPS_BYTE else "⚠️  memory-bound"
    print(f"  {T:>8,}  {kv_gb:>9.2f}GB  {read_time_ms:>10.2f} ms  "
          f"{intensity:>11.2f}   {regime}")

print(f"\n  Every generation step is memory-bound at all practical context lengths.")
print(f"  The KV cache helps computation — but makes the memory wall worse.")

In [ ]:
# ── Mini roofline with our actual LLM operating points ──

fig, ax = plt.subplots(figsize=(10, 5.5))

I_range = np.logspace(-1, 4, 400)
roof    = np.minimum(PEAK_TFLOPS * np.ones_like(I_range),
                     BANDWIDTH_GBps * 1e-3 * I_range)

ax.plot(I_range, roof, color=ACCENT2, linewidth=3, label="A100 Roofline (FP16)")
ax.axvline(RIDGE_OPS_BYTE, color=YELLOW, linewidth=1.5, linestyle="--",
           label=f"Ridge point  ({RIDGE_OPS_BYTE:.0f} ops/byte)")
ax.fill_between(I_range, 0, roof, where=(I_range < RIDGE_OPS_BYTE),
                alpha=0.12, color=WARN)
ax.fill_between(I_range, 0, roof, where=(I_range >= RIDGE_OPS_BYTE),
                alpha=0.12, color=OK)

colors_pts = plt.cm.plasma(np.linspace(0.3, 0.9, len(context_lengths)))
for i, T in enumerate(context_lengths):
    kv_bytes  = 2 * T * d_k_gpt3 * n_heads_gpt3 * n_layers_gpt3 * dtype_bytes
    flops_step= 2 * T * d_k_gpt3 * n_heads_gpt3 * n_layers_gpt3
    intensity = flops_step / kv_bytes
    perf      = min(PEAK_TFLOPS, BANDWIDTH_GBps * 1e-3 * intensity)
    ax.scatter(intensity, perf, color=colors_pts[i], s=120, zorder=5)
    ax.annotate(f"T={T}", (intensity, perf),
                textcoords="offset points", xytext=(5, 4),
                fontsize=8, color=colors_pts[i])

ax.set_xscale("log")
ax.set_xlabel("Arithmetic Intensity  [ops / byte]", fontsize=11)
ax.set_ylabel("Achieved Performance  [TFLOP/s]", fontsize=11)
ax.set_title("GPT-3 Scale Inference on A100 — All Context Lengths Are Memory-Bound",
             fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)
fig.tight_layout()
plt.savefig("roofline_gpt3.png", dpi=140, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("📊 Every single operating point is far left of the ridge. The GPU is starved for data.")

## 8. 🎯 Final Output — The Complete Story in One Dashboard

Let us produce the definitive visual summary of everything we have learned.

In [ ]:
# ── THE GRAND SUMMARY DASHBOARD ──

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor("#0a0a14")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# ── A: Naive vs cached FLOPs (log scale) ──
ax_a = fig.add_subplot(gs[0, 0])
T_range = np.arange(10, 2001, 20)
ax_a.plot(T_range, [4 * t**2 * d_k_gpt3 / 1e9 for t in T_range],
          color=WARN, linewidth=2.5, label="Naive  O(T²)")
ax_a.plot(T_range, [4 * t * d_k_gpt3 / 1e9 for t in T_range],
          color=OK,   linewidth=2.5, label="Cached O(T)")
ax_a.set_yscale("log")
ax_a.set_xlabel("Context length T")
ax_a.set_ylabel("FLOPs per step (×10⁹)")
ax_a.set_title("A: Computation Cost", fontsize=11)
ax_a.legend(fontsize=8)
ax_a.grid(True)

# ── B: Roofline schematic ──
ax_b = fig.add_subplot(gs[0, 1])
I_b   = np.logspace(-1, 4, 300)
roof_b = np.minimum(PEAK_TFLOPS, BANDWIDTH_GBps * 1e-3 * I_b)
ax_b.plot(I_b, roof_b, color=ACCENT2, linewidth=3)
ax_b.axvline(RIDGE_OPS_BYTE, color=YELLOW, linewidth=1.5, linestyle="--")
ax_b.fill_between(I_b, 0, roof_b, where=(I_b < RIDGE_OPS_BYTE),
                  alpha=0.2, color=WARN, label="Memory-Bound")
ax_b.fill_between(I_b, 0, roof_b, where=(I_b >= RIDGE_OPS_BYTE),
                  alpha=0.2, color=OK, label="Compute-Bound")
ax_b.scatter([1.0], [BANDWIDTH_GBps * 1e-3 * 1.0], color=WARN, s=180,
             zorder=5, label="KV-cached LLM")
ax_b.set_xscale("log")
ax_b.set_xlabel("Arithmetic Intensity (ops/byte)")
ax_b.set_ylabel("Performance (TFLOP/s)")
ax_b.set_title("B: The Roofline Model", fontsize=11)
ax_b.legend(fontsize=8)
ax_b.grid(True)

# ── C: KV cache VRAM by model size ──
ax_c = fig.add_subplot(gs[0, 2])
ctx_lengths = [512, 1024, 2048, 4096, 8192]
configs = {
    "GPT-2 (small)":   (12, 12, 64),
    "GPT-3 (175B)":    (96, 96, 128),
    "LLaMA-2 (70B)":   (80, 64, 128),
}
bar_w = 0.25
x_pos = np.arange(len(ctx_lengths))
for i, (name, (nl, nh, dk)) in enumerate(configs.items()):
    vrams = [2 * T * dk * nh * nl * 2 / 1e9 for T in ctx_lengths]
    ax_c.bar(x_pos + i * bar_w, vrams, bar_w, label=name, alpha=0.85)
ax_c.set_xticks(x_pos + bar_w)
ax_c.set_xticklabels([str(T) for T in ctx_lengths], fontsize=8)
ax_c.set_xlabel("Context length")
ax_c.set_ylabel("KV Cache VRAM (GB)")
ax_c.set_title("C: VRAM Cost by Model Size", fontsize=11)
ax_c.legend(fontsize=7)
ax_c.grid(True, axis="y")

# ── D: Attention weight pattern (causal) ──
ax_d = fig.add_subplot(gs[1, 0])
T_d = 20
Q_d = torch.randn(T_d, 32)
K_d = torch.randn(T_d, 32)
scores_d = Q_d @ K_d.T / 32**0.5
mask_d   = torch.triu(torch.ones(T_d, T_d, dtype=torch.bool), diagonal=1)
scores_d = scores_d.masked_fill(mask_d, float("-inf"))
weights_d = torch.softmax(scores_d, dim=-1).detach().numpy()
im_d = ax_d.imshow(weights_d, cmap="magma", aspect="auto")
ax_d.set_xlabel("Key position (cache index)")
ax_d.set_ylabel("Query (generation step)")
ax_d.set_title("D: Causal Attention Pattern", fontsize=11)
plt.colorbar(im_d, ax=ax_d)

# ── E: Bandwidth pressure vs context ──
ax_e = fig.add_subplot(gs[1, 1])
T_e_range = np.arange(128, 8193, 128)
kv_bytes_e  = [2 * T * 128 * 64 * 32 * 2 for T in T_e_range]  # LLaMA-ish
read_time_e = [b / (BANDWIDTH_GBps * 1e9) * 1000 for b in kv_bytes_e]
ax_e.plot(T_e_range, read_time_e, color=YELLOW, linewidth=2.5)
ax_e.fill_between(T_e_range, 0, read_time_e, alpha=0.15, color=YELLOW)
ax_e.set_xlabel("Context length (tokens)")
ax_e.set_ylabel("Min time to read KV cache (ms)")
ax_e.set_title("E: Memory Read Time Floor", fontsize=11)
ax_e.grid(True)

# ── F: Summary table ──
ax_f = fig.add_subplot(gs[1, 2])
ax_f.axis("off")
table_data = [
    ["", "Without Cache", "With KV Cache"],
    ["FLOPs / step", "O(T²·d_k)", "O(T·d_k)"],
    ["Bytes read", "O(T·d_k)", "O(T·d_k)"],
    ["Arith. Intensity", "O(T)", "O(1)  ← lower!"],
    ["Bottleneck", "Compute", "Memory BW"],
    ["Latency scaling", "Quadratic", "Linear"],
    ["VRAM cost", "Low", "Grows as O(T)"],
]
table = ax_f.table(cellText=table_data[1:],
                   colLabels=table_data[0],
                   cellLoc="center", loc="center",
                   bbox=[0, 0.1, 1, 0.85])
table.auto_set_font_size(False)
table.set_fontsize(8.5)
for (row, col), cell in table.get_celld().items():
    cell.set_facecolor("#1a1a2e")
    cell.set_edgecolor("#444")
    cell.set_text_props(color="white")
    if row == 0:
        cell.set_facecolor(ACCENT)
    if col == 2 and row > 0:
        cell.set_facecolor("#0d2b0d")
ax_f.set_title("F: Summary", fontsize=11, color="white")

fig.suptitle(
    "🧠  The KV Cache — Brilliant Fix, Hidden Cost\n"
    "Vizuara  ·  LLM Systems Series",
    fontsize=14, color="white", y=1.01, fontweight="bold"
)

plt.savefig("kv_cache_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("🎉 Dashboard saved! This is your screenshot-worthy final output.")

## 9. Reflection and Next Steps

### 🤔 Reflection Questions

Take a moment to think through these — they are designed to consolidate what you have built.

**On the mechanism:**
1. We said that Keys and Values for past tokens "never change." Under what architectural assumption is this exactly true? (Hint: think about what happens if the model uses a different kind of positional encoding, like RoPE applied inside the attention head.)

2. In our implementation, `attention_with_cache` gets the new token's Query, Key, and Value as inputs. In a real transformer, where do these come from? What weight matrices are involved, and do *those* computations get cached too?

**On the hardware:**
3. We showed that the KV cache lowers arithmetic intensity. But look at panel E of the dashboard — even just *reading* the cache takes measurable milliseconds. What would happen if instead of one user, you had 64 users sending requests simultaneously? Would the memory bottleneck get better or worse? (This is called *batching*, and it is the key to increasing arithmetic intensity.)

4. Our roofline model treats bandwidth as a fixed constant. In reality, what factors might cause the effective bandwidth to be much lower than the hardware's peak? (Think about cache lines, DRAM access patterns, and the difference between sequential and random reads.)

**On the trade-off:**
5. For a model serving 10,000 requests per minute, each with a 4,096-token context: estimate how much GPU memory is consumed by KV caches alone (use GPT-3 scale numbers from our bandwidth analysis). How does this compare to the model weights themselves?

### 🏆 Optional Challenges

Ready to go deeper? Here are three challenges in increasing order of difficulty.

**Challenge 1 — Multi-Head KV Cache** *(Moderate)*
Extend the `KVCache` class to handle multiple attention heads simultaneously. Your new class should accept a `n_heads` parameter and maintain separate caches per head. Verify that the output matches the single-head version when `n_heads=1`.

**Challenge 2 — Sliding Window Cache** *(Hard)*
Real systems like Mistral use a *sliding window* — the cache only retains the most recent $W$ tokens rather than the full context. Implement this as a `SlidingWindowKVCache` that drops the oldest entry when the cache exceeds size $W$. Plot how accuracy (measured as cosine similarity to the full-cache output) degrades as $W$ decreases relative to context length.

**Challenge 3 — Arithmetic Intensity Sweep on Real Hardware** *(Advanced)*
Use `torch.cuda.Event` to precisely time the GPU operations in `attention_no_cache` and `attention_with_cache` for sequence lengths from 64 to 4096. Plot your measured throughput (FLOPs/s, computed using theoretical FLOPs and measured time) on a real roofline diagram for your GPU (look up your GPU's specs). Do your measurements confirm the memory-bound prediction?

In [ ]:
# ── A clean summary of the key numbers from this notebook ──

print("=" * 65)
print("  🎓  KEY TAKEAWAYS — The KV Cache")
print("=" * 65)
print()
print("  1. PROBLEM")
print("     Naive auto-regressive attention costs O(T²·d) FLOPs per step.")
print("     At T=2000 this is catastrophically slow.")
print()
print("  2. THE FIX — KV Cache")
print("     Store K and V vectors for past tokens once, reuse forever.")
print("     Reduces per-step FLOPs from O(T²) → O(T).")
print()
print("  3. THE HIDDEN COST")
print("     Reading the cache from VRAM costs O(T) bytes every step.")
print("     Arithmetic intensity I = FLOPs/bytes drops to ~O(1).")
print("     We slide deep into the MEMORY-BOUND regime of the roofline.")
print()
print("  4. THE NUMBERS (A100, GPT-3 scale, T=2048)")
kv_bytes_final = 2 * 2048 * 128 * 96 * 96 * 2 / 1e9
read_ms_final  = kv_bytes_final / 2_000 * 1_000
print(f"     KV Cache size   : {kv_bytes_final:.1f} GB")
print(f"     Read time floor : {read_ms_final:.1f} ms  (just to fetch data!)")
print(f"     Ridge point     : {RIDGE_OPS_BYTE:.0f} ops/byte")
print(f"     Our intensity   : ~1 ops/byte  →  {RIDGE_OPS_BYTE:.0f}× below ridge")
print()
print("  5. WHAT COMES NEXT")
print("     → Paged attention (vLLM): manage cache like virtual memory")
print("     → Multi-Query Attention : share K,V across heads → fewer bytes")
print("     → Flash Attention       : fuse ops to stay in fast SRAM")
print("     → Speculative decoding  : generate multiple tokens per step")
print()
print("  Thanks for working through this with us — Vizuara ✨")
print("=" * 65)

---

*You have just built the KV cache from first principles, measured its computational cost, and understood exactly where it sits on the GPU's roofline. You are now equipped to read papers on paged attention, multi-query attention, and speculative decoding with a concrete mental model of the problem they are solving.*

*In the next notebook in this series, we will look at **Paged Attention** — how vLLM treats the KV cache like virtual memory to serve thousands of users simultaneously without wasting a single byte of VRAM.*

*— The Vizuara Team*